In [0]:
movie_df = (
    spark.read
    .table("workspace.default.movies")
)

display(movie_df)

In [0]:
from pyspark.sql import functions as F

# Get the list of distinct genres by splitting the "genres" column on "|"
distinct_genres = (
    movie_df
    .select(F.explode(F.split(F.col("genres"), "\\|")).alias("genre"))
    .distinct()
    .orderBy("genre")
)

genre_list = [row["genre"] for row in distinct_genres.collect()]
print(f"Found {len(genre_list)} distinct genres: {genre_list}")

# Build one boolean column per genre
movies_genre_df = movie_df
for genre in genre_list:
    col_name = f"genre_{genre.replace('-', '_').replace(' ', '_').lower()}"
    movies_genre_df = movies_genre_df.withColumn(
        col_name,
        F.array_contains(F.split(F.col("genres"), "\\|"), genre)
    )

# Drop the original string column since each genre now has its own column
movies_genre_df = movies_genre_df.drop("genres")

display(movies_genre_df)

In [0]:
from pyspark.sql import functions as F

# Extract the 4-digit year in parentheses at the end of the title, e.g. "Movie Name (2002)"
movies_genre_df = movies_genre_df.withColumn(
    "release_year",
    F.regexp_extract(F.col("title"), "\\((\\d{4})\\)\\s*$", 1).cast("int")
)

# Optionally clean the year out of the title, leaving just the movie name
movies_genre_df = movies_genre_df.withColumn(
    "title",
    F.regexp_replace(F.col("title"), "\\s*\\(\\d{4}\\)\\s*$", "")
)

display(movies_genre_df)


In [0]:
movie_genre_silver_table = (
    movies_genre_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.movie_genre_silver")
)